In [1]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType, FloatType

In [2]:
spark = SparkSession.builder \
    .appName("MySQL JDBC Example") \
    .config("spark.jars", "/Users/nikkinani/Downloads/mysql-connector-j-8.2.0/mysql-connector-j-8.2.0.jar") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

26/01/09 13:18:36 WARN Utils: Your hostname, Nikkis-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.17.65.4 instead (on interface en0)
26/01/09 13:18:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/01/09 13:18:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/09 13:18:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
df = spark.read.load('/Users/nikkinani/Documents/googleData/googleplaystore.csv', format='csv', header='true', inferSchema='true',escape = '"')
df.printSchema()

root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Reviews: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)



In [4]:
df.count()

10841

In [5]:
df.show(10, False)

+--------------------------------------------------+--------------+------+-------+----+-----------+----+-----+--------------+-------------------------+------------------+------------------+------------+
|App                                               |Category      |Rating|Reviews|Size|Installs   |Type|Price|Content Rating|Genres                   |Last Updated      |Current Ver       |Android Ver |
+--------------------------------------------------+--------------+------+-------+----+-----------+----+-----+--------------+-------------------------+------------------+------------------+------------+
|Photo Editor & Candy Camera & Grid & ScrapBook    |ART_AND_DESIGN|4.1   |159    |19M |10,000+    |Free|0    |Everyone      |Art & Design             |January 7, 2018   |1.0.0             |4.0.3 and up|
|Coloring book moana                               |ART_AND_DESIGN|3.9   |967    |14M |500,000+   |Free|0    |Everyone      |Art & Design;Pretend Play|January 15, 2018  |2.0.0             

In [6]:
df = df.withColumn("Reviews", col("Reviews").cast(IntegerType())) \
    .withColumn("Installs", regexp_replace(col("Installs"),"[^0-9]","")) \
    .withColumn("Price" , regexp_replace(col("Price"),"[$]","")) \
    .withColumn("Installs", col("Installs").cast(IntegerType())) \
    .withColumn("Price", col("Price").cast(FloatType()))

In [7]:
# get The top 10 highest reviewed app from the data frame
highest_reviewed_apps = df .groupby("App") \
                           .agg(sum("Reviews").alias("Total Reviews")) \
                           .orderBy(desc("Total Reviews")) \
                           .show(10, False)

+----------------------------------------+-------------+
|App                                     |Total Reviews|
+----------------------------------------+-------------+
|Instagram                               |266241989    |
|WhatsApp Messenger                      |207348304    |
|Clash of Clans                          |179558781    |
|Messenger – Text and Video Chat for Free|169932272    |
|Subway Surfers                          |166331958    |
|Candy Crush Saga                        |156993136    |
|Facebook                                |156286514    |
|8 Ball Pool                             |99386198     |
|Clash Royale                            |92530298     |
|Snapchat                                |68045010     |
+----------------------------------------+-------------+
only showing top 10 rows



In [9]:
# category wise highest installed apps

wdw = Window.partitionBy("Category").orderBy(sum("Installs").desc)

df_ranked = df.withColumn("rank", rank().over(wdw))

top3_apps = df_ranked.filter(col("rank") <= 3)

top3_apps.select("Category", "App", "Installs", "rank").show(truncate=False)

PySparkTypeError: [NOT_COLUMN_OR_STR] Argument `col` should be a Column or str, got method.